# **Clinical Trial Enrichment Pipeline (Production v03)**
This notebook orchestrates the high-fidelity clinical enrichment of 44,000 trials using Gemini 2.0 Flash and IHME GBD taxonomy.

# **1. Environment Setup & Anti-Leakage Sanitization**

In this stage, we initialize the analytical environment. 
- **Dependencies**: We load `google-genai` for Gemini 2.0 interaction and `tqdm` for progress tracking.
- **Project Sanitizer**: We import the `day_zero_reconstructor`. This is a critical custom module that removes timestamps and post-registration results from clinical text, ensuring the model only sees information available on the trial's start date (Day Zero).

In [1]:
import pandas as pd
import os
import csv
import re
import sys
import json
import time
import asyncio
import logging
from tqdm.asyncio import tqdm
from dotenv import load_dotenv
from google import genai
from google.genai import types

# [STEP 1] Load environment variables from .env (API Keys)
load_dotenv()

# [STEP 2] Link the local source code folder so we can use the 'day_zero_reconstructor'
sys.path.append('..')
from src.prep.text_cleaning import day_zero_reconstructor

# [STEP 3] Define the working directories for raw data and processed results
DATA_PATH = '../data/'
OUTPUT_PATH = '../data/processed'

# [STEP 4] Global Newline Helper: Bypasses JSON parsing issues in notebook strings
NL = chr(10)

# [STEP 5] Utility function for robust CSV loading with specific clinical formatting
def safe_load(filename, cols=None):
    full_path = os.path.join(DATA_PATH, filename)
    params = {
        "sep": "|", "dtype": str, "header": 0, "quotechar": '"',
        "quoting": 3, "low_memory": False, "on_bad_lines": "warn"
    }
    return pd.read_csv(full_path, usecols=cols, **params)

print("> SUCCESS: Environment Ready.")


> SUCCESS: Environment Ready.


# **2. Production Filtering Gates & Temporal Priority**

We define the 44,000-trial universe by applying strict clinical filters.
- **Sorting**: We sort by **Start Date (Descending)** so that the most recent clinical innovations (2024–2026) are processed first.
- **Filters**: We restrict the cohort to industry-led, interventional Phase 2/3 drug trials.

In [2]:
# [STEP 1] Load core trial metadata (status, type, phase, dates)
df = safe_load('studies.txt', cols=['nct_id', 'overall_status', 'study_type', 'phase', 'start_date', 'official_title', 'brief_title', 'why_stopped', 'number_of_arms'])
print(f"Initial: {len(df)} trials.")

# [STEP 2] Fill missing Official Title with Brief Title (matching data_loader_01 logic)
df['official_title'] = df['official_title'].fillna(df['brief_title'])

# [STEP 3] Filter for Industry-led trials ONLY (excluding Academic/NIH for high-stakes modeling)
df_sponsors = safe_load('sponsors.txt', cols=['nct_id', 'lead_or_collaborator', 'agency_class', 'name'])
industry_ids = df_sponsors[(df_sponsors['lead_or_collaborator'].str.upper() == 'LEAD') & (df_sponsors['agency_class'].str.upper() == 'INDUSTRY')]['nct_id'].unique()
df = df[df['nct_id'].isin(industry_ids) & (df['study_type'].str.upper() == 'INTERVENTIONAL')]

# [STEP 4] Filter for "Valley of Death" phases (Phase 2 & 3) - excluding safety/post-market
excluded_phases = ['EARLY_PHASE1', 'PHASE1', 'PHASE4', 'NA']
df = df[~df['phase'].fillna('NA').str.upper().isin(excluded_phases)]

# [STEP 5] Apply Temporal Sorting: Process most recent clinical innovations (2005-2026) first
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df = df[df['start_date'].dt.year.between(2005, 2026)].sort_values('start_date', ascending=False)

# [STEP 6] COVID-19 Sanitizer: Remove trials that failed due to pandemic logistics (not scientific failure)
covid_pat = r'covid|pandemic|coronavirus|sars-cov-2'
df = df[~df['why_stopped'].fillna('').str.lower().str.contains(covid_pat, regex=True)]

print(f"> Filtered Universe: {len(df)} trials. (Sorted: Most Recent First)")


Initial: 564443 trials.
> Filtered Universe: 46238 trials. (Sorted: Most Recent First)


# **3. Cross-File Evidence Harvesting**

Clinical signals are scattered across multiple tables. We aggregate evidence from MeSH terms, Study arms, Interventions, and Primary Outcomes to build a unified evidence base for the LLM.

In [3]:
# [STEP 1] Select a randomized test cohort (300 trials) for the audit run
trial_df = df.sample(30, random_state=100).copy()
target_ids = trial_df['nct_id'].tolist()

# [STEP 2] Load and optimize all evidence sources (Raw Disease Names prioritized over MeSH)
df_cond = safe_load('conditions.txt', cols=['nct_id', 'name'])
df_int = safe_load('interventions.txt', cols=['nct_id', 'intervention_type', 'name', 'description'])
df_arms = safe_load('design_groups.txt', cols=['nct_id', 'group_type', 'title', 'description'])
df_sum = safe_load('brief_summaries.txt', cols=['nct_id', 'description'])
df_elig = safe_load('eligibilities.txt', cols=['nct_id', 'criteria'])

# [STEP 3] Harmonize Outcomes: Merge Primary endpoints from both the standard and design tables
df_out1 = safe_load('outcomes.txt', cols=['nct_id', 'outcome_type', 'title', 'time_frame'])
df_out2 = safe_load('design_outcomes.txt', cols=['nct_id', 'outcome_type', 'measure', 'time_frame'])
df_out2 = df_out2.rename(columns={'measure': 'title'})
df_outcomes = pd.concat([df_out1, df_out2], ignore_index=True)

# [STEP 4] OPTIMIZATION: Convert massive tables into Fast-Access Dictionaries (O(1) lookups)
print(">>> Building Fast-Access Dictionaries (Pre-calculating 150k+ rows)...")
from collections import defaultdict

def create_lookup(df, col_name=None):
    # This pre-sorts data into a Map {NCT_ID -> [List of Records]}
    lookup = defaultdict(list)
    for _, row in df.iterrows():
        lookup[row['nct_id']].append(row if col_name is None else row[col_name])
    return lookup

# Pre-calculate all lookups so the main loop runs in seconds, not hours
cond_lookup = create_lookup(df_cond, 'name')
int_lookup = create_lookup(df_int)
arms_lookup = create_lookup(df_arms)
sum_lookup = {row['nct_id']: row['description'] for _, row in df_sum.iterrows()}
elig_lookup = {row['nct_id']: row['criteria'] for _, row in df_elig.iterrows()}
outcomes_lookup = create_lookup(df_outcomes)

# Optimize Lead Sponsor lookup for quick identity retrieval
df_leads = df_sponsors[df_sponsors['lead_or_collaborator'].str.upper() == 'LEAD']
leads_lookup = {row['nct_id']: row['name'] for _, row in df_leads.iterrows()}

print(f"> Evidence Harvested & O(1) Dictionaries Ready for {len(target_ids)} trials.")


>>> Building Fast-Access Dictionaries (Pre-calculating 150k+ rows)...
> Evidence Harvested & O(1) Dictionaries Ready for 30 trials.


In [4]:
results = []
for nct_id in target_ids:
    # [STEP 1] Retrieve the specific metadata for this trial
    row = trial_df[trial_df['nct_id'] == nct_id].iloc[0]

    # [STEP 2] Identity & Sanitization: Remove future timestamps/results to prevent data leakage
    clean_title = day_zero_reconstructor(row['official_title'], "title")
    sponsor_raw = leads_lookup.get(nct_id, "Unknown")

    # [STEP 3] Arm Evidence: Format study groups (Placebo vs Experimental) into readable blocks
    arm_blocks = []
    for arm in arms_lookup.get(nct_id, []):
        d = day_zero_reconstructor(arm['description'], "arm") if pd.notna(arm['description']) else "No description"
        arm_blocks.append(f"ARM_TYPE: {arm['group_type']} | TITLE: {arm['title']}" + NL + f"ARM_DESC: {d}")
    arms_string = (NL + "---" + NL).join(arm_blocks)

    # [STEP 4] Intervention Evidence: Extract drug names and mechanism of action descriptions
    int_blocks = []
    for drug in int_lookup.get(nct_id, []):
        d = day_zero_reconstructor(drug['description'], "intervention") if pd.notna(drug['description']) else "No description"
        int_blocks.append(f"NAME: {drug['name']} ({drug['intervention_type']})" + NL + f"DESC: {d}")
    ints_string = (NL + "---" + NL).join(int_blocks)

    # [STEP 5] Disease Information: Use Raw Names (conditions.txt) for maximum scientific nuance
    conds = " | ".join(list(set(cond_lookup.get(nct_id, []))))

    # [STEP 6] Protocol Essence: Clean the brief summary for LLM context window efficiency
    raw_summary = sum_lookup.get(nct_id, "")
    clean_summary = day_zero_reconstructor(raw_summary, "summary")

    # [STEP 7] Eligibility Rules: Include FULL Inclusion + Exclusion criteria (matching BioBERT input)
    raw_criteria = elig_lookup.get(nct_id, "")
    clean_criteria = day_zero_reconstructor(raw_criteria, "criteria")

    # [STEP 8] Primary Outcomes: Extract objective measures (PFS, OS, etc.) and timeframes
    out_blocks = []
    seen_outcomes = set()
    for out in outcomes_lookup.get(nct_id, []):
        ot = str(out['outcome_type']).lower()
        title = str(out['title']).strip()
        if 'primary' in ot and title not in seen_outcomes:
            m = day_zero_reconstructor(title, "outcome")
            t = day_zero_reconstructor(str(out['time_frame']), "timeframe")
            out_blocks.append(f"TITLE: {m} | TIMEFRAME: {t}")
            seen_outcomes.add(title)
    outcomes_string = (NL + "---" + NL).join(out_blocks[:10])

    # [STEP 9] Rare Disease Detector: Flag potential orphan indications before sending to LLM
    rare_pattern = r'\b(rare|orphan|ultra-rare|niche disease)\b'
    has_rare_hint = "YES" if re.search(rare_pattern, raw_summary.lower()) or re.search(rare_pattern, row['official_title'].lower()) else "NO"

    # [STEP 10] Final Super-Context Assembly: The "One-Shot" Dossier that the AI will read
    # ADDED: [PHASE] and [NUMBER_OF_ARMS] for high-alpha field anchoring
    context_body = f"""[TRIAL_START]
[NCT_ID]: {nct_id}
[START_YEAR]: {row['start_date'].year}
[PHASE]: {row['phase']}
[NUMBER_OF_ARMS]: {row['number_of_arms']}
[OFFICIAL_TITLE]: {clean_title}
[RAW_LEAD_SPONSOR]: {sponsor_raw}
[RARE_DISEASE_HINT]: {has_rare_hint}

[STUDY_ARMS_AND_COMPARATORS]:
{arms_string}

[TRIAL_CONDITIONS_RAW]:
{conds}

[PRIMARY_ENDPOINTS_DETAIL]:
{outcomes_string}

[INTERVENTION_DETAILS]:
{ints_string}

[PROTOCOL_SUMMARY]:
{clean_summary[:1200]}...

[ELIGIBILITY_CRITERIA_FULL]:
{clean_criteria[:2500]}...
[TRIAL_END]"""

    results.append({"nct_id": nct_id, "context": context_body})

# [STEP 11] Save the assembled context blocks to a physical file for the enrichment stage
pd.DataFrame(results).to_csv(os.path.join(OUTPUT_PATH, 'research_context.csv'), index=False)
print(f"> Success: Assembled context for {len(results)} trials using O(1) lookups and Raw Names.")


> Success: Assembled context for 30 trials using O(1) lookups and Raw Names.


## **Shift to terminal and run:** 
# python3 src/prep/enrichment_runner.py